In [ ]:
%pip install pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, round, lit, log10, max as _max, min as _min
import pandas as pd

# 1. Setup & Ingestion
# Read from workspace file using pandas, then convert to Spark
workspace_file = "/Workspace/Users/zuzanna1872@gmail.com/hpc_hno_2025.csv"
# Skip row 1 (index 1) which contains HXL tags
pandas_df = pd.read_csv(workspace_file, dtype=str, skiprows=[1])

# Convert to Spark DataFrame, Spark will handle the type inference
df = spark.createDataFrame(pandas_df)

# Removing HXL tags and filtering for 'total' category to avoid double-counting demographics
df_clean = df.filter(~col("Country ISO3").contains("#")) \
    .filter(col("Category").ilike("total")) \
    .withColumn("In_Need", col("In Need").cast("double")) \
    .withColumn("Targeted", col("Targeted").cast("double")) \
    .withColumn("Population", col("Population").cast("double")) \
    .filter(col("In_Need").isNotNull())

# 2. Normalization & Severity (Urgency)
# Severity Index: What percentage of the local population is in need?
df_metrics = df_clean.withColumn(
    "Severity_Index", 
    round(col("In_Need") / when(col("Population") > 0, col("Population")).otherwise(lit(1)), 4)
)

# 3. Planning Gap (A core component of Neglect)
# If Targeted is low relative to In_Need, the crisis is structurally neglected
df_metrics = df_metrics.withColumn(
    "Planning_Gap",
    round(1 - (when(col("Targeted") > 0, col("Targeted")).otherwise(0) / col("In_Need")), 4)
)

# 4. Importance (Scale)
# We use a log scale for 'Importance' so that massive crises don't completely 
# drown out smaller, high-severity crises in the math.
df_metrics = df_metrics.withColumn(
    "Importance_Score",
    round(log10(col("In_Need") + 1), 2)
)

# 5. The Lighthouse Priority Score
# Combined Score = Severity (Urgency) * Planning Gap (Neglect) * Importance (Scale)
df_metrics = df_metrics.withColumn(
    "Lighthouse_Priority",
    round(col("Severity_Index") * col("Planning_Gap") * col("Importance_Score"), 4)
)

# 6. Rank and Prepare for Dashboard
final_analysis = df_metrics.select(
    col("Country ISO3").alias("Country_ISO3"),
    "Cluster", 
    "In_Need", 
    "Population", 
    "Severity_Index", 
    "Planning_Gap", 
    "Importance_Score", 
    "Lighthouse_Priority"
).orderBy(col("Lighthouse_Priority").desc())

# Create schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS lighthouse_os")

# Save for Databricks SQL Dashboarding
final_analysis.write.mode("overwrite").saveAsTable("lighthouse_os.decision_metrics")

display(final_analysis)